# Building the baseline dataset

### Reminder: Setting-up and accessing Google Cloud Platform (Big Query, Cloud Storage) via the client

**Installing packages** 

* virtualenv (recomended for installing google-cloud packages)
    * https://packaging.python.org/guides/installing-using-pip-and-virtual-environments/
    * https://janakiev.com/blog/jupyter-virtual-envs/ This article gives good instructions how to create a virtual environment with Anaconda and how to add it in your Jupyter notebook

* google-cloud-bigquery 
    * https://cloud.google.com/bigquery/docs/reference/libraries 
    
**Creating a virtual environment with Anaconda and adding it to Jupyter** 

1. Open Anaconda prompt
2. _conda create -n myenv_  where myenv is any name you want to set for your virtual environment, it is stored in the envs folder in your Anaconda directory. 
3. To start working in it: _conda activate myenv_; 
    * to stop: _conda deactivate_
    * to list all available environments: _conda env list_
    * to remove an environment: conda env list _conda env remove -n myenv_
4. After activating the virtual environment, you need to add it to Jupyter. First, install ipykernel which provides the IPython kernel for Jupyter _pip install --user ipykernel_
5. This command will add the environment to Jupyter _python -m ipykernel install --user --name=myenv_
6. You should see the following output _Installed kernelspec myenv in /home/user/.local/share/jupyter/kernels/myenv_
7. Now you are able to choose this new environment as a kernel in Jupyter 
    * In an open notebook: Kernel --> Change Kernel --> myenv
    * Once you remove the virtual environment, you can remove the kernel from Jupyter: _jupyter kernelspec uninstall myenv_
8. Again from Anaconda prompt (make sure you are in the virtual environment): 
    * _pip install google-cloud-bigquery_
    * _pip install google-cloud-storage_
    Note: if you install storage later than bigquery, you might run into some conflicts (different versions of basic packages). To solve: run _pip install --upgrade google-cloud-core_, _pip install --upgrade google-resumable-media_, _pip install --upgrade google-cloud-bigquery_ --user. Depending on your pandas installation, you might need to _pip install db-dtypes --user_. 

**Setting up authentication on Google Cloud** (you can do the same using the command line)
1. In the Cloud Console, select the relevant project and go to the Create service account key page.  https://console.cloud.google.com/apis/credentials/serviceaccountkey 
2. From the Service account list, select New service account. In the Service account name field, enter a name. From the Role list, select Project > Owner.
3. Save the generated json file

In [ ]:
# Import packages
!pip install google-cloud-bigquery
!pip install --upgrade google-cloud-bigquery[pandas]
import os, sys
from google.cloud import bigquery
from google.cloud import storage
import pandas as pd
from pathlib import Path

In [ ]:
# Set path
ROOT = Path(__file__).resolve().parents[2]

DATA = ROOT / "database"
EXTRACTIONS = DATA / "extractions"
OUTPUT = ROOT / "output"
FIGURES = OUTPUT / "figures"
TABLES = OUTPUT / "tables"

In [ ]:
# Credentials 
credentials_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")

if not credentials_path:
    raise EnvironmentError(
        "GOOGLE_APPLICATION_CREDENTIALS is not set. "
        "Please set it to the path of your Google Cloud credentials JSON file."
    )

credentials_path = Path(credentials_path).expanduser().resolve()

if not credentials_path.exists():
    raise FileNotFoundError(
        f"Google Cloud credentials file not found: {credentials_path}"
    )

In [ ]:
# Construct a BigQuery client object.
client = bigquery.Client()
storage_client = storage.Client()

In [ ]:
# settings for bigquery 
dry = bigquery.QueryJobConfig(dry_run = True, use_query_cache=False) 
run = bigquery.QueryJobConfig(dry_run=False, use_query_cache=True)

## Reminder: Github datasets 
GitHub queries
* https://github.blog/2017-01-19-github-data-ready-for-you-to-explore-with-bigquery/?fbclid=IwAR1E01NhM1kFZE4TM_XC6aDhkWSm2s8oCIsKXA4EcsiixnNdsBo22Kjlwho 

* https://github.com/fhoffa/analyzing_github/

* Note: there are currently several GitHub datasets available on BigQuery
    * GHtorrent data: ghtorrentmysql1906 - contains the most available GHtorrent dump
    * GHtorrent data 2: ghtorrent-bq - GHtorrent dumps from 2017 and 2018 https://ghtorrent.org/ 
    * GitHub Activity data: bigquery-public-data:github_repos. Contains contents from 2.9M public, open source licensed repositories on GitHub. https://console.cloud.google.com/marketplace/details/github/github-repos?filter=solution-type:dataset&q=github&id=46ee22ab-2ca4-4750-81a7-3ee0f0150dcb
    * GitHub Archive data: githubarchive Contains data on GitHub events. https://www.gharchive.org/

## Dataset
0. run a simple summary query (country pair) for commits in 2012 and 2013 to select the relevant control group for UA and RU users
1. create an updated user table (base: GHtorrent 2019 edition)
    * drop fake users (bots to submit commits)
    * merge together all available information on user locations (user_id, year, reported location in this year)
    * result: user_panel/all_locations
2. select unique user_ids from UA, RU, control countries
    * result: user_panel/user_ids
3. select commits by users from UA, RU, control aggregated by month/year, baseline sample: 2012-2018 (2 years before and 5 years after the conflict)
    * author_id, project_id, month, year number of commits 
    * result: commits/commits
    * note: project owners are not restricted to control countries only. 
4. select info from all relevant projects (project ids extracted from step 3 + their first parents if available)
    * project_id, owner_id, url, name, language, created_at, deleted, upated_at, forked_from
    * relevant infromation about first "parents" of forked projects 
    * result: commits/projects
5. select users from step 2, add users (who are project owners in step 4 and not covered in step 2) - generate full users table
    * result: commits/users_sample
6. export users_sample, commits, and projects from BQ to Google Storage, then download csv files from Google Storage to the local drive
7. summary tables for additional controls
    * country_code, month, number of users 
    * country_code, month, number of projects 
    * country_code of owner_id, country code of user_id (watcher), month, number of watches (=stars)
    * country_code of follower_id, country code of user_id, month, number of followers 
    * result: download directly and save locally to csv files 

### User locations notes

* User panel for 2016-2021 (based on already geocoded location with a country code), credit to Lena Abou El-Komboz
* For locations 2013-2015 http://ghtorrent-downloads.ewi.tudelft.nl/mysql/ 
* Downloaded files, imported on google mysql server and then to google big query. Attention: expensive to store, delete after extracting user tables and the relevant information 
* Help: https://hoffa.medium.com/loading-mysql-backup-files-into-bigquery-straight-from-cloud-sql-d40a98281229
    * On Google Cloud: go to SQL server, create instance, choose 'Development', storage location 'US central' (should be the same as the location of data on Google Storage and connection on BigQuery), create a user (note password), create a new database. 
    * Import data from the storage (MySQL files loaded to Google storage beforehand). Note: it can take several hours. 
    * Go to BigQuery --> Add data --> External data source --> specify details from the loaded database
    
        Run this query
        
            CREATE TABLE `friends-foes.user_panel.users2013` 
             AS 
            SELECT * FROM EXTERNAL_QUERY("friends-foes.us.gh20131012", "SELECT * FROM users;");
   


In [ ]:
# In case tables expire, use this code to reload from GS to BigQuery
# users2013 (extracted from MySQL to not keep the whole file)
# users2014 (extracted from MySQL to not keep the whole file)

In [ ]:
# users 2016
# load a table from GS to bigquery
table_id = "friends-foes.user_panel.users2016"
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id", "INTEGER"),
        bigquery.SchemaField("name", "STRING"), 
        bigquery.SchemaField("email", "STRING"), 
        bigquery.SchemaField("login", "STRING"), 
        bigquery.SchemaField("company", "STRING"), 
        bigquery.SchemaField("created_at", "STRING"), 
        bigquery.SchemaField("location", "STRING"), 
        bigquery.SchemaField("country_code", "STRING"), 
        bigquery.SchemaField("city", "STRING"),
        bigquery.SchemaField("lat", "STRING"),
        bigquery.SchemaField("lng", "STRING"), 
        bigquery.SchemaField("form_address", "STRING")
    ],
    skip_leading_rows=1,
    field_delimiter = ',', 
    ignore_unknown_values = True, 
    max_bad_records = 1000
    )

uri = "gs://github_user_datasets/users2016.csv"

load_job = client.load_table_from_uri(
    uri, table_id, job_config=job_config
)  # Make an API request.

load_job.result()  # Wait for the job to complete.

table = client.get_table(table_id)
print("Loaded {} rows to table {}".format(table.num_rows, table_id))

In [ ]:
# ghpanel
table_id = "friends-foes.user_panel.ghuserpanel"
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("counter", "INTEGER"),
        bigquery.SchemaField("table_year", "INTEGER"),
        bigquery.SchemaField("id", "INTEGER"), 
        bigquery.SchemaField("login", "STRING"), 
        bigquery.SchemaField("fake", "INTEGER"),
        bigquery.SchemaField("long", "STRING"), 
        bigquery.SchemaField("lat", "STRING"),
        bigquery.SchemaField("country_code", "STRING"),
        bigquery.SchemaField("nunique", "INTEGER")
    ],
    skip_leading_rows=1,
    field_delimiter = ',', 
    ignore_unknown_values = True, 
    max_bad_records = 1000
    )

uri = "gs://github_user_datasets/GHUserPanel.csv"

load_job = client.load_table_from_uri(
    uri, table_id, job_config=job_config
)  # Make an API request.

load_job.result()  # Wait for the job to complete.

table = client.get_table(table_id)
print("Loaded {} rows to table {}".format(table.num_rows, table_id))

In [ ]:
# 0. Summary statistics: activity by country pair before January 2014 to motivate the composition of the control group
# Use locations as of 2016 (because 2013/2014 are not geocoded yet )
query = """
SELECT year, country_author, country_owner, SUM(gh_events) AS gh_events, 
COUNT(DISTINCT project_id) AS active_projects, COUNT(DISTINCT author_login) AS active_contributors
  FROM
  (SELECT *
  FROM 
    (SELECT EXTRACT(YEAR FROM created_at) AS year, author_id, project_id, COUNT(*) AS gh_events
    FROM `ghtorrentmysql1906.MySQL1906.commits` 
    WHERE EXTRACT(YEAR FROM created_at) < 2014
    GROUP BY year, author_id, project_id
    ) commits
  INNER JOIN 
    (SELECT owner_id, id FROM `ghtorrentmysql1906.MySQL1906.projects`) projects
  ON commits.project_id = projects.id
  # Take out author or owner country code null
  INNER JOIN (SELECT  id as author_id, login as author_login, country_code AS country_author
              FROM    `friends-foes.user_panel.users2016`
              WHERE   country_code IS NOT NULL) author
  ON commits.author_id = author.author_id
  INNER JOIN (SELECT  id as owner_id, country_code AS country_owner
              FROM     `friends-foes.user_panel.users2016`
              WHERE   country_code IS NOT NULL) owner
  ON projects.owner_id = owner.owner_id
  WHERE commits.author_id != projects.owner_id #owner and committer are different
  )
GROUP BY year, country_author, country_owner  
"""

In [ ]:
# check processing costs
job = client.query(query, job_config=dry)
print("Total GB that will be processed: ", job.total_bytes_processed/1000000000)
print("Bytes billed: ", job.total_bytes_billed)

In [ ]:
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
sum_country_pairs = job.to_dataframe()  
sum_country_pairs.head()

In [ ]:
country_pairs = sum_country_pairs[sum_country_pairs['year'] > 2011] # start the sample from 2012
country_pairs = country_pairs[country_pairs['country_author']!="\\N"] # drop non-reported countries

In [ ]:
# Can see if it changes if we add 
# lower bound of country-pair contributors, because some are using locations, which are not geocoded /assigned to a country
partner_ru = country_pairs[country_pairs['country_owner'].str.contains('ru')].groupby(['country_author', 'country_owner'], as_index=False).agg({"gh_events":"sum", "active_projects":"sum", 'active_contributors':"sum"}).sort_values(by = ['active_contributors'], ascending = False).head(15)
partner_ua = country_pairs[country_pairs['country_owner'].str.contains('ua')].groupby(['country_author', 'country_owner'], as_index=False).agg({"gh_events":"sum", "active_projects":"sum", 'active_contributors':"sum"}).sort_values(by = ['active_contributors'], ascending = False).head(15)
# 15. biggest contributors to Russian projects + those in the previous version
# 15 biggest contributors to Ukrainian projects + those in the previous version
# adding them to be able to control for RUproject*time, UAproject*time fixed effects 
print(partner_ua['country_author'].unique())
print(partner_ru['country_author'].unique())

In [ ]:
# Check if the same countries are also the biggest destinations for UA/RU programmers
partner_rua = country_pairs[country_pairs['country_author'].str.contains('ru')].groupby(['country_author', 'country_owner'], as_index=False).agg({"gh_events":"sum", "active_projects":"sum", 'active_contributors':"sum"}).sort_values(by = ['active_contributors'], ascending = False).head(15)
partner_uaa = country_pairs[country_pairs['country_author'].str.contains('ua')].groupby(['country_author', 'country_owner'], as_index=False).agg({"gh_events":"sum", "active_projects":"sum", 'active_contributors':"sum"}).sort_values(by = ['active_contributors'], ascending = False).head(15)
# 15. biggest contributors to Russian projects + those in the previous version
# 15 biggest contributors to Ukrainian projects + those in the previous version
# adding them to be able to control for RUproject*time, UAproject*time fixed effects 
print(partner_uaa['country_owner'].unique())
print(partner_rua['country_owner'].unique())

In [ ]:
# countries are in the list of 20 biggest contributor to UA/RU in 2012-2013, used for control
pd.concat([partner_ua, partner_ru])['country_author'].unique()

In [ ]:
# before running all locations table, remove duplicates from 2016 table 
query = """
DROP VIEW IF EXISTS `friends-foes.user_panel.users2016d`;
CREATE VIEW `friends-foes.user_panel.users2016d` 
AS 
(
SELECT DISTINCT *
FROM
(SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY lng DESC) as n
FROM `friends-foes.user_panel.users2016`
)
WHERE n = 1
)
"""
client.query(query, job_config=run) 

In [ ]:
# 1. all_locations table in BQ (panel)
# NOTE: order of columns matter for UNION !!! 
# NOTE: might not run from Jupyter because of \\N in line 63, but the same query executed directly in BQ works well!
# NOTE: users2013, users2014, users2016 and ghpanelusers might need to be uploaded again to BQ from Google Storage 

query = """
DROP TABLE IF EXISTS `friends-foes.user_panel.all_users_locations_panel`;

CREATE TABLE `friends-foes.user_panel.all_users_locations_panel` AS
SELECT DISTINCT id, 2019 AS year, STRING(NULL) as name, STRING(NULL) as email, company, created_at, type, deleted, 
       CAST(long as STRING) as long, CAST(lat as STRING) as lat, country_code, city, location
 FROM 
    (SELECT * FROM `ghtorrentmysql1906.MySQL1906.users` WHERE fake = 0) #locations as of 2019
  UNION ALL 
    (SELECT DISTINCT id, 2013 AS year, name, email, company, created_at, type, NULL as deleted, 
            STRING(NULL) as long, STRING(NULL) as lat, STRING(NULL) as country_code, STRING(NULL) as city, location 
    FROM `friends-foes.user_panel.users2013`)  #locations as of 2013, locations not geocoded!
  UNION ALL 
    (SELECT DISTINCT id, 2014 AS year, name, email, company, created_at, type, NULL as deleted, 
            STRING(NULL) as long, STRING(NULL) as lat, STRING(NULL) as country_code, STRING(NULL) as city, location 
    FROM `friends-foes.user_panel.users2014`)  #locations as of 2014, locations not geocoded!  
  UNION ALL #data from Thomas' panel (2016-2018) + data on locations from 2016 (users2016)
    (
     SELECT 
      CASE
         WHEN u2016.id IS NOT NULL THEN u2016.id
         ELSE panel.id
      END id, 
      CASE
         WHEN u2016.year IS NOT NULL THEN u2016.year
         ELSE panel.year
      END year,
      name, email, company, NULL as created_at, NULL as type, NULL as deleted,
      CASE
        WHEN u2016.long IS NOT NULL THEN CAST(u2016.long as STRING)
        ELSE CAST(panel.long as STRING)
      END long,
      CASE
        WHEN u2016.lat IS NOT NULL THEN CAST(u2016.lat as STRING)
        ELSE CAST(panel.lat as STRING)
      END lat,
      CASE
        WHEN u2016.country_code IS NOT NULL THEN u2016.country_code
        ELSE panel.country_code
      END country_code,
     city, location 
     FROM
        (SELECT DISTINCT id, CAST(SUBSTRING(CAST(table_year as STRING), 1, 4) as INT64) as year,
        # select first values, because there are several editions per year 
          FIRST_VALUE(country_code)
          OVER (PARTITION BY CAST(CAST(id as STRING)||SUBSTRING(CAST(table_year as STRING), 1, 4) as INT64)
                ORDER BY table_year) AS country_code, 
          FIRST_VALUE(long)
          OVER (PARTITION BY CAST(CAST(id as STRING)||SUBSTRING(CAST(table_year as STRING), 1, 4) as INT64)
                ORDER BY table_year) AS long, 
          FIRST_VALUE(lat)
          OVER (PARTITION BY CAST(CAST(id as STRING)||SUBSTRING(CAST(table_year as STRING), 1, 4) as INT64)
                ORDER BY table_year) AS lat 
        FROM `friends-foes.user_panel.ghuserpanel` 
        WHERE CAST(SUBSTRING(CAST(table_year as STRING), 1, 4) as INT64) BETWEEN 2016 AND 2018) panel # drop 2019, because we already have full data from 2019 dumps
      FULL JOIN 
       (SELECT DISTINCT id, 2016 AS year, name, email, company, lng AS long, lat, 
       CASE 
          WHEN country_code = "\\N" THEN NULL
          ELSE country_code
        END country_code,
        city, location 
        FROM `friends-foes.user_panel.users2016d`) u2016 #to have locations info for 2016
      ON u2016.id = panel.id AND u2016.year = panel.year
    )
    
"""
# commented out to not repeat the load 
client.query(query, job_config=run) 


In [ ]:
# 2(0) upload a table with UA and RU major cities in English, Russian, and Ukrainian 
# in stata: data_prep
# upload to google storage 
bucket_name =  'github_user_datasets'
destination_blob_name = 'cities_ua_ru.txt'
source_file_name = 'database/extractions/cities_ua_ru_cloud.txt'

bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(destination_blob_name)
blob.upload_from_filename(source_file_name)

print(
        f"File {source_file_name} uploaded to {destination_blob_name}."
)
    

In [ ]:
# load a table from GS to bigquery
table_id = "friends-foes.user_panel.cities_ua_ru"
job_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("city", "STRING"),
        bigquery.SchemaField("country", "STRING"), 
        bigquery.SchemaField("longitude", "FLOAT"),
        bigquery.SchemaField("latitude", "FLOAT"), 
        bigquery.SchemaField("language", "STRING")
    ],
    skip_leading_rows=1,
    field_delimiter = '\t'
    )

uri = "gs://github_user_datasets/cities_ua_ru.txt"

load_job = client.load_table_from_uri(
    uri, table_id, job_config=job_config
)  # Make an API request.

load_job.result()  # Wait for the job to complete.

table = client.get_table(table_id)
print("Loaded {} rows to table {}".format(table.num_rows, table_id))

#maybe for later: fuzzy string matching https://towardsdatascience.com/a-journey-into-bigquery-fuzzy-matching-4-of-1-a-tribute-to-fuzzywuzzy-54ff73e91a1c

In [ ]:
# 2. select distinct ids from the relevant countries (use all_locations table)
# note: for some reasons different logins can link to the same id.. (few cases)
query = """
DROP TABLE IF EXISTS `friends-foes.user_panel.user_idsv2`;
CREATE TABLE `friends-foes.user_panel.user_idsv2` AS 
SELECT DISTINCT id
FROM `friends-foes.user_panel.all_users_locations_panel`
WHERE 
# based on country codes (from 2016 and later)
    country_code IN ('ua', 'us', 'ru', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'pl', 'jp',
       'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee')
# based on country codes in location
    OR lower(location) IN ('ua', 'us', 'ru', 'de', 'gb', 'fr', 'ca', 'au', 'nl', 'pl', 'jp',
       'cn', 'se', 'in', 'br', 'it', 'by', 'cz', 'lt', 'lv', 'ee')
# UA users 
   OR lower(location) IN ('kyiv', 'kiev', 'київ', 'киев', 'kharkiv', 'kharkov', 'харків', 'харьков', 
    'odessa', 'odesa', 'одесса', 'одеса', 'dnipropetrovsk', 'dnepropetrovsk', 'дніпропетровськ', 'днепропетровск', 
    'donetsk', 'донецк', 'донецьк', 'zaporizhzhya', 'zaporozhje', 'запорожье', 'запоріжжа', 'lviv', 'lvov', 'львів', 'львов', 'cherkasy','черкасы', 'vinnytsya',  'вінниця', 'винница', 'krym', 'sevastopol', 'simferopol', 'yalta', 'крым', 'севастополь', 'симферополь', 'ялта',
    'kherson', 'херсон', 'mykolaiv', 'миколаїв', 'николаев', 'luhansk', 'луганск')
    OR lower(city) IN ('kyiv', 'kiev', 'київ', 'киев', 'kharkiv', 'kharkov', 'харків', 'харьков', 
    'odessa', 'odesa', 'одесса', 'одеса', 'dnipropetrovsk', 'dnepropetrovsk', 'дніпропетровськ', 'днепропетровск', 
    'donetsk', 'донецк', 'донецьк', 'zaporizhzhya', 'zaporozhje', 'запорожье', 'запоріжжа', 'lviv', 'lvov', 'львів', 'львов', 'cherkasy','черкасы', 'vinnytsya',  'вінниця', 'винница', 'krym', 'sevastopol', 'simferopol', 'yalta', 'крым', 'севастополь', 'симферополь', 'ялта',
    'kherson', 'херсон', 'mykolaiv', 'миколаїв', 'николаев', 'luhansk', 'луганск')
    OR lower(location) IN
            (SELECT city FROM `friends-foes.user_panel.cities_ua_ru` WHERE country = 'ua')
# RU users 
    OR lower(location) IN ('moscow', 'saint petersburg', 'novosibirsk', 'yekaterinburg', 'kazan', 'samara', 
        'tomsk', 'nizhniy novgorod', 'krasnodar', 'москва', 'санкт-петербург', 'новосибирск', 'екатеринбург', 
        'казань', 'самара', 'томск', 'нижний новгород', 'краснодар')
    OR lower(city) IN ('moscow', 'saint petersburg', 'novosibirsk', 'yekaterinburg', 'kazan', 'samara', 
        'tomsk', 'nizhniy novgorod', 'krasnodar', 'москва', 'санкт-петербург', 'новосибирск', 'екатеринбург', 
        'казань', 'самара', 'томск', 'нижний новгород', 'краснодар')
    OR lower(location) IN
            (SELECT city FROM `friends-foes.user_panel.cities_ua_ru` WHERE country = 'ru')
    OR lower(location) LIKE '%укр%' OR lower(location) LIKE '%ukra%' OR lower(location) LIKE '%ucra%'
    OR lower(location) LIKE '%russi%' OR lower(location) LIKE '%rossi%' OR lower(location) LIKE '%рф%' OR lower(location) LIKE '%росс%'
"""
# commented out to not repeat the load 
client.query(query, job_config=run) 

In [ ]:
# 3a.client.query(query, job_config=run)  extract project ids owned by relevant users 
query = """
DROP VIEW IF exists `friends-foes.commits.projects-sample`;

CREATE VIEW `friends-foes.commits.projects-sample`
AS
SELECT DISTINCT id, owner_id FROM 
`ghtorrentmysql1906.MySQL1906.projects` 
WHERE owner_id IN (SELECT id FROM `friends-foes.user_panel.user_idsv2`)
"""
client.query(query, job_config=run) 

In [ ]:
# 3b. extract commits from the relevant users and from the relevant projects
query = """
DROP TABLE IF EXISTS `friends-foes.commits.commits`;
CREATE TABLE `friends-foes.commits.commits` AS 
SELECT 
    EXTRACT(YEAR FROM created_at) AS year, 
    EXTRACT(MONTH FROM created_at) AS month, 
    author_id, project_id, COUNT(*) AS commits
FROM `ghtorrentmysql1906.MySQL1906.commits` 
WHERE 
(
    author_id IN 
    (SELECT DISTINCT id FROM `friends-foes.user_panel.user_idsv2`)
    OR 
    project_id IN 
    (SELECT DISTINCT id FROM `friends-foes.commits.projects-sample`)
)
    AND
    EXTRACT(YEAR FROM created_at) BETWEEN 2010 AND 2018
GROUP BY year, month, author_id, project_id
"""
client.query(query, job_config=run) 
# the resulting file is too big to download it directly as a dataframe here: save as a BQ table, export to Cloud Storage, download locally from Cloud Storage
# total number of observations  61 073 206

In [ ]:
# 4. extract all relevant project info from commits: repositories of a commit + first parents for those that were forked 
query = """
DROP TABLE IF EXISTS `friends-foes.commits.projects`;
CREATE TABLE `friends-foes.commits.projects` AS
SELECT DISTINCT id, owner_id, name, description, language, created_at, forked_from, deleted FROM `ghtorrentmysql1906.MySQL1906.projects` 
WHERE id IN
      (
          (SELECT DISTINCT  project_id from `friends-foes.commits.commits`)
          UNION DISTINCT 
          (SELECT DISTINCT forked_from AS project_id 
              FROM `ghtorrentmysql1906.MySQL1906.projects`
              WHERE id IN 
                (SELECT DISTINCT project_id from `friends-foes.commits.commits`)
          )
      )
"""
client.query(query, job_config=run) 

In [ ]:
# 5. extract all relevant users with their locations: those from user_ids + those who are owners of projects from the commits table 
query = """
DROP TABLE IF EXISTS `friends-foes.commits.users_sample`;

CREATE TABLE `friends-foes.commits.users_sample` AS
SELECT DISTINCT id, year, name, email, company, created_at, type, deleted, long, lat, 
       country_code, city, location, 
      # adding country codes for UA and RU users without geocoded data
    CASE 
      WHEN 
        lower(location) IN ('ua', 'kyiv', 'kiev', 'київ', 'киев', 'kharkiv', 'kharkov', 'харків', 'харьков', 
      'odessa', 'odesa', 'одесса', 'одеса', 'dnipropetrovsk', 'dnepropetrovsk', 'дніпропетровськ', 'днепропетровск', 
      'donetsk', 'донецк', 'донецьк', 'zaporizhzhya', 'zaporozhje', 'запорожье', 'запоріжжа', 'lviv', 'lvov', 'львів', 'львов',
      'cherkasy','черкасы', 'vinnytsya', 'вінниця', 'винница', 'krym', 'sevastopol', 'simferopol', 'yalta', 'крым', 'севастополь', 'симферополь', 'ялта',
      'kherson', 'херсон', 'mykolaiv', 'миколаїв', 'николаев', 'luhansk', 'луганск' )
        OR lower(city) IN ('ua', 'kyiv', 'kiev', 'київ', 'киев', 'kharkiv', 'kharkov', 'харків', 'харьков', 
      'odessa', 'odesa', 'одесса', 'одеса', 'dnipropetrovsk', 'dnepropetrovsk', 'дніпропетровськ', 'днепропетровск', 
      'donetsk', 'донецк', 'донецьк', 'zaporizhzhya', 'zaporozhje', 'запорожье', 'запоріжжа', 'lviv', 'lvov', 'львів', 'львов',
      'cherkasy','черкасы', 'vinnytsya', 'вінниця', 'винница', 'krym', 'sevastopol', 'simferopol', 'yalta', 'крым', 'севастополь', 'симферополь', 'ялта',
      'kherson', 'херсон', 'mykolaiv', 'миколаїв', 'николаев', 'luhansk', 'луганск' )
        OR lower(location) LIKE '%укр%' OR lower(location) LIKE '%ukra%' OR lower(location) LIKE '%ucra%'
        OR lower(location) IN
            (SELECT city FROM `friends-foes.user_panel.cities_ua_ru` WHERE country = 'ua')
      THEN 'ua'
      WHEN lower(location) IN ('ru', 'moscow', 'saint petersburg', 'novosibirsk', 'yekaterinburg', 'kazan', 'samara', 
        'tomsk', 'nizhniy novgorod', 'krasnodar', 'москва', 'санкт-петербург', 'новосибирск', 'екатеринбург', 
        'казань', 'самара', 'томск', 'нижний новгород', 'краснодар')
      OR lower(city) IN ('ru', 'moscow', 'saint petersburg', 'novosibirsk', 'yekaterinburg', 'kazan', 'samara', 
        'tomsk', 'nizhniy novgorod', 'krasnodar', 'москва', 'санкт-петербург', 'новосибирск', 'екатеринбург', 
        'казань', 'самара', 'томск', 'нижний новгород', 'краснодар')
      OR lower(location) LIKE '%russi%' OR lower(location) LIKE '%rossi%' OR lower(location) LIKE '%рф%' OR lower(location) LIKE '%росс%'
      OR lower(location) IN
            (SELECT city FROM `friends-foes.user_panel.cities_ua_ru` WHERE country = 'ru')
      THEN 'ru'
      ELSE country_code
    END country_code_f
FROM `friends-foes.user_panel.all_users_locations_panel`
WHERE id IN
      (
          (SELECT DISTINCT author_id from `friends-foes.commits.commits`)
          UNION DISTINCT
          (SELECT DISTINCT owner_id AS id from `friends-foes.commits.projects`)
      )
"""
client.query(query, job_config=run) 

In [ ]:
# 6a. create a balanced banel panel with user locations 
query = """
DROP TABLE IF EXISTS `friends-foes.user_panel.sample_panel`;

CREATE TABLE `friends-foes.user_panel.sample_panel`
AS
SELECT p.id, p.year, year_created, month_created, cc2019, long2019, lat2019, loc2019, 
      cc2016, long2016, lat2016, loc2016, name2016,
      cc2013, loc2013, name2013,
      cc2014, loc2014, name2014,
      cc_current, loc_current, long_current, lat_current
FROM 
        (SELECT id, year, year_created, month_created, cc2019, long2019, lat2019, loc2019
                FROM
            # 1. select user ids and static info as of 2019
            # note: for some observations created_at as of 2019 is wrongly (too late) recorded! 
            # e.g. user_id 3375545 - created_at in 2018, but also have non-missing info for 2014-2016... 
        (SELECT a.id,  
          CASE WHEN min_year < EXTRACT(year from created_at) THEN min_year ELSE EXTRACT(year from created_at) END year_created, 
          EXTRACT(MONTH FROM created_at) AS month_created, country_code_f AS cc2019, 
          long AS long2019, lat AS lat2019, location as loc2019 
          FROM (SELECT * FROM `friends-foes.commits.users_sample` WHERE year = 2019 AND created_at IS NOT NULL) a
          JOIN 
          (SELECT DISTINCT id, MIN(year) OVER(PARTITION BY id) as min_year
            FROM `friends-foes.user_panel.all_users_locations_panel`) b
            ON a.id = b.id  
        ) 
            # 2. create a panel for users using years from commits table 
                 CROSS JOIN 
                (SELECT * FROM UNNEST([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]) AS year)
          WHERE year >= year_created
          ) p
            LEFT JOIN 
            # 3. for all users and years, attach location as of 2016
            (SELECT DISTINCT id, country_code_f as cc2016, long AS long2016, lat AS lat2016, location as loc2016, name as name2016
                FROM `friends-foes.commits.users_sample` WHERE year = 2016) u16
            ON p.id = u16.id
            # 4. for all users and years, attach location as of 2013 and 2014
            LEFT JOIN 
            (SELECT DISTINCT id, country_code_f as cc2013, location as loc2013, name as name2013
                FROM `friends-foes.commits.users_sample` WHERE year = 2013) u13
            ON p.id = u13.id
            LEFT JOIN 
            (SELECT DISTINCT id, country_code_f as cc2014, location as loc2014, name as name2014
                FROM `friends-foes.commits.users_sample` WHERE year = 2014) u14
            ON p.id = u14.id
            # 5. for all users and years, attach their current location (from Thomas' data)
            LEFT JOIN 
            (SELECT DISTINCT id, year, country_code_f AS cc_current, long AS long_current, lat AS lat_current, location AS loc_current
                FROM `friends-foes.commits.users_sample`) act
            ON p.id = act.id AND p.year = act.year
"""
client.query(query, job_config=run) 

In [ ]:
# 6b. create a panel with user locations, replacing misisng locations
# NOTE!!!! Might not run from Jyputer because of \\N -- the code works well in the Cloud
# NOTE: missing country codes before 2014 are replaced with 2013 data, 2016 are replaced with 2016 data, missing country codes in 2017-18 are replaced with 201
query = """
DROP TABLE IF EXISTS `friends-foes.user_panel.sample_locations_panel`; 

CREATE TABLE `friends-foes.user_panel.sample_locations_panel` AS 

#intrapolate missing locations with first non-missing reported
SELECT DISTINCT id, year_created, month_created, year, 
cc2013, cc2014, cc2016, cc2019, loc2013, loc2014, loc2016, loc2019, 
name2013, name2014, name2016,
# replace remaining missing  with 2019
CASE WHEN (country_current IS NULL OR country_current IN ('NA', '\\N')) THEN cc2019 ELSE country_current END country_current,
CASE WHEN (long_current IS NULL OR long_current IN ('NA', '\\N')) THEN long2019  ELSE long_current END long_current,
CASE WHEN (lat_current IS NULL OR lat_current IN ('NA', '\\N')) THEN lat2019  ELSE lat_current END lat_current,
CASE WHEN (loc_current IS NULL OR loc_current IN ('NA', '\\N')) THEN loc2019  ELSE loc_current END loc_current
FROM 
( #replace remaining missing  with 2016
    SELECT id,  year_created, month_created, year, 
            cc2013, cc2014, cc2016, cc2019, loc2013, loc2014, loc2016, loc2019, 
            name2013, name2014, name2016, long2019, lat2019,
            CASE WHEN year < 2016 AND country_current IS NULL THEN cc2016 ELSE country_current END country_current,
            CASE WHEN year < 2016 AND long_current IS NULL THEN long2016  ELSE long_current    END long_current,
            CASE WHEN year < 2016 AND lat_current IS NULL THEN lat2016    ELSE lat_current     END lat_current,
            CASE WHEN year < 2016 AND loc_current IS NULL THEN loc2016    ELSE loc_current     END loc_current
    FROM 
    (   # replace missing locations before 2014 with 2014 data (if 2013 is not available), save as m
        SELECT id, year_created, month_created, year, 
            cc2013, cc2014, cc2016, cc2019, loc2013, loc2014, loc2016, loc2019, 
            name2013, name2014, name2016, 
            long_current, lat_current, long2016, lat2016, long2019, lat2019,
            # replace missing before 2014 with 2014 
            CASE WHEN year < 2014 AND country_current IS NULL THEN cc2014  ELSE country_current END country_current,
            CASE WHEN year < 2014 AND loc_current IS NULL THEN loc2014 ELSE loc_current END loc_current
        FROM 
        (# replace missing locations before 2013 with 2013 data (no earlier is available), save as m
        SELECT id,  year_created, month_created, year, 
            cc2013, cc2014, cc2016, cc2019, loc2013, loc2014, loc2016, loc2019,
            long_current, lat_current, long2016, lat2016, long2019, lat2019,
            name2013, name2014, name2016,
            # replace missing before 2013 with 2013 
            CASE WHEN year < 2013 AND cc_current IS NULL THEN cc2013 ELSE cc_current END country_current,
            CASE WHEN year < 2013 AND loc_current IS NULL THEN loc2013 ELSE loc_current END loc_current
        FROM `friends-foes.user_panel.sample_panel`
        )
    )
)
"""
client.query(query, job_config=run) 

In [ ]:
#remove duplicates from cities_ua_ru
query = """
DROP VIEW IF EXISTS `friends-foes.user_panel.cities_ua_rud`;
CREATE VIEW `friends-foes.user_panel.cities_ua_rud` 
AS 
(
SELECT DISTINCT *
FROM
(SELECT *, ROW_NUMBER() OVER (PARTITION BY city ORDER BY country DESC) as n
FROM `friends-foes.user_panel.cities_ua_ru`
)
WHERE n = 1
)
"""
client.query(query, job_config=run) 

In [ ]:
# 7. export UA users to link them to the UA admin data and assign identitites (see 02_UA_users for the rest)
query = """
SELECT *  
FROM 
(SELECT *
FROM `friends-foes.user_panel.sample_locations_panel` 
WHERE country_current = 'ua'
) a
LEFT JOIN `friends-foes.user_panel.cities_ua_rud` b
ON lower(a.loc_current) = b.city
"""
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
ua_users = job.to_dataframe()  
ua_users.to_csv(EXTRACTIONS / 'ua_users.csv', sep = '|', encoding = 'utf-8', index=False)  

In [ ]:
# 8.  select aggregated data for summary statistics 
# number of registered users, by country code (first reported country), year, month
query = """
SELECT country_code, 
       EXTRACT(YEAR FROM created_at) AS year, 
       EXTRACT(MONTH FROM created_at) AS month,  
       COUNT(*) AS new_users
FROM `friends-foes.user_panel.all_users_locations_panel`
WHERE year = 2019
GROUP BY country_code, year, month
"""
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
sum_users = job.to_dataframe()  
sum_users.to_csv(EXTRACTIONS /  'sum_users_country.csv', sep = '|', encoding = 'utf-8', index=False)

In [ ]:
# number of registered projects, by country code (first reported country), year, month
query = """
SELECT country_code, 
       year, 
       month,  
       SUM(new_projects) AS new_projects 
FROM 
    (SELECT EXTRACT(YEAR FROM created_at) AS year, 
            EXTRACT(MONTH FROM created_at) AS month, 
            owner_id, 
            COUNT(*) AS new_projects
     FROM `ghtorrentmysql1906.MySQL1906.projects`
     GROUP BY year, month, owner_id
     ) projects 
LEFT JOIN 
    (SELECT id, country_code FROM `friends-foes.user_panel.all_users_locations_panel` WHERE year = 2019) users
ON projects.owner_id = users.id
GROUP BY country_code, year, month
"""
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
sum_users = job.to_dataframe()  
sum_users.to_csv(EXTRACTIONS / 'sum_projects_country.csv', sep = '|', encoding = 'utf-8', index=False)

In [ ]:
# number of stars (in the table watchers) received by projects, by country pair, year, month
query = """
SELECT owner_country, 
       watcher_country, 
       EXTRACT(YEAR FROM created_at) AS year,
       EXTRACT(MONTH FROM created_at) AS month, 
       COUNT(*) AS watchers_month
FROM `ghtorrentmysql1906.MySQL1906.watchers`
JOIN 
    (SELECT id, country_code AS watcher_country 
    FROM `friends-foes.user_panel.all_users_locations_panel` WHERE year = 2019 #AND country_code NOT IN ('ru', 'ua')
    ) watchers
ON user_id = watchers.id
JOIN
    (SELECT p.id AS project_id, country_code AS owner_country
    FROM `ghtorrentmysql1906.MySQL1906.projects` p
    JOIN 
    (SELECT * FROM `friends-foes.user_panel.all_users_locations_panel` WHERE year = 2019) u
    ON p.owner_id = u.id
    )
ON repo_id = project_id
GROUP BY owner_country, watcher_country, year, month
"""
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
sum_users = job.to_dataframe()  
sum_users.to_csv(EXTRACTIONS / 'sum_watchers.csv', sep = '|', encoding = 'utf-8', index=False)

In [ ]:
# number of followers received by user, by country pair, year, month (followers are not in RU/UA)
# number of stars (in the table watchers) received by projects 
query = """
SELECT user_country, 
       follower_country, 
       EXTRACT(YEAR FROM created_at) AS year,
       EXTRACT(MONTH FROM created_at) AS month, 
       COUNT(*) AS followers_month
FROM `ghtorrentmysql1906.MySQL1906.followers`
JOIN 
    (SELECT id, country_code AS user_country 
    FROM `friends-foes.user_panel.all_users_locations_panel` 
    WHERE year = 2019 
    ) users
ON user_id = users.id
JOIN
    (SELECT id, country_code AS follower_country 
    FROM `friends-foes.user_panel.all_users_locations_panel` 
    WHERE year = 2019 AND country_code NOT IN ('ru', 'ua')
    ) users2
ON follower_id = users2.id
GROUP BY user_country, follower_country, year, month
"""
job = client.query(query, job_config=run)  #comment it out if not needed to repeat
sum_users = job.to_dataframe()  
sum_users.to_csv(EXTRACTIONS / 'sum_followers.csv', sep = '|', encoding = 'utf-8', index=False)